# Two-stage Portfolio Decision-Support Pipeline

This notebook was generated from the current two_stage_portfolio_pipeline.py file. It keeps the edited configuration, function names, and data flow, while replacing argparse/main execution with notebook cells.


## 1. Imports and Configuration

Edit paths, tau, model types, target names, and Top-k ratios in this cell.


In [1]:
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils import shuffle

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier, XGBRegressor
except ImportError:
    XGBClassifier = None
    XGBRegressor = None

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None


RANDOM_SEED = 42
SCREENING_THRESHOLD = 0.5
CLASSIFIER_TYPE = "xgboost"
PROFIT_MODEL_TYPE = "mlr"
TOP_K_RATIOS = [0.10, 0.20, 0.30]

TRAIN_DATA_PATH = "train.csv"
TEST_DATA_PATH = "test.csv"
SYNTHETIC_DATA_PATH = "vae-ctabgan.csv"
RESULTS_DIR = "results"

TARGET_COL = "loan_status"
DEFAULT_LABEL = 1
RETURN_COL = "annualized_return"
PROFIT_COL = "realized_profit"

LABELENCODING_FEATURES = ["sub_grade"]
ONEHOT_FEATURES = ["home_ownership", "purpose"]

SCREENING_DROP_FEATURES = [
    # Post-loan repayment and recovery information
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp',
    'total_rec_int', 'total_rec_late_fee',
    'out_prncp', 'out_prncp_inv',
    'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt',
    'next_pymnt_d',

    # Post-origination credit information
    'last_credit_pull_d',
    'last_fico_range_high', 'last_fico_range_low',

    # Realized investment outcomes
    'return', 'annualized_return', 'realized_profit',

    # Debt settlement outcomes
    'debt_settlement_flag', 'debt_settlement_flag_date',
    'settlement_status', 'settlement_date',
    'settlement_amount', 'settlement_percentage', 'settlement_term',

    # Used for temporal split or profit calculation, not for screening features
    'issue_d',
    'funded_amnt_inv',
]


PROFIT_DROP_FEATURES = [
    "loan_status",
    "total_pymnt_inv",
    "funded_amnt_inv",
    "fico_range_high",
    "return",
    "annualized_return",
    "realized_profit",
]

from types import SimpleNamespace

args = SimpleNamespace(
    train_path=TRAIN_DATA_PATH,
    test_path=TEST_DATA_PATH,
    synthetic_path=SYNTHETIC_DATA_PATH,
    results_dir=RESULTS_DIR,
    seed=RANDOM_SEED,
    tau=SCREENING_THRESHOLD,
    classifier_type=CLASSIFIER_TYPE,
    profit_model_type=PROFIT_MODEL_TYPE,
    top_k_ratios=TOP_K_RATIOS,
    target_col=TARGET_COL,
    return_col=RETURN_COL,
    profit_col=PROFIT_COL,
)

args


namespace(train_path='train.csv',
          test_path='test.csv',
          synthetic_path='vae-ctabgan.csv',
          results_dir='results',
          seed=42,
          tau=0.5,
          classifier_type='xgboost',
          profit_model_type='mlr',
          top_k_ratios=[0.1, 0.2, 0.3],
          target_col='loan_status',
          return_col='annualized_return',
          profit_col='realized_profit')

## 2. Pipeline Functions

These functions are copied from the current Python script, excluding parse_args and main.


In [2]:
def require_columns(df, columns, name):
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required column(s): {missing}")


def add_realized_outcomes(df, return_col=RETURN_COL, profit_col=PROFIT_COL):
    require_columns(df, ["total_pymnt_inv", "funded_amnt_inv", "term_months"], "data")
    df = df.copy()
    df["return"] = (df["total_pymnt_inv"] - df["funded_amnt_inv"]) / df["funded_amnt_inv"]
    df["annualized_return"] = (
        df["total_pymnt_inv"] / df["funded_amnt_inv"]
    ) ** (12 / df["term_months"]) - 1
    df["realized_profit"] = df["total_pymnt_inv"] - df["funded_amnt_inv"]
    df = df.replace([np.inf, -np.inf], np.nan)

    require_columns(df, [return_col, profit_col], "data after outcome construction")
    return df


def fit_encoders(df, label_cols, onehot_cols):
    require_columns(df, label_cols + onehot_cols, "training data")

    label_encoders = {}
    for col in label_cols:
        le = LabelEncoder()
        le.fit(df[col])
        label_encoders[col] = le

    onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    onehot_encoder.fit(df[onehot_cols])
    return label_encoders, onehot_encoder


def encode_features(df, label_encoders, onehot_encoder, label_cols, onehot_cols):
    df = df.copy()
    require_columns(df, label_cols + onehot_cols, "data to encode")

    for col in label_cols:
        le = label_encoders[col]
        known_values = set(le.classes_)
        df[col] = df[col].map(lambda x: x if x in known_values else le.classes_[0])
        df[col] = le.transform(df[col])

    onehot_encoded = onehot_encoder.transform(df[onehot_cols])
    onehot_df = pd.DataFrame(
        onehot_encoded,
        columns=onehot_encoder.get_feature_names_out(onehot_cols),
        index=df.index,
    )

    df.drop(columns=onehot_cols, inplace=True)
    df = pd.concat([df, onehot_df], axis=1)
    return df


def get_classifier(classifier_type, seed):
    if classifier_type == "xgboost":
        if XGBClassifier is None:
            raise ImportError("xgboost is required for classifier_type='xgboost'.")
        return XGBClassifier(
            n_estimators=500,
            learning_rate=0.01,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=seed,
        )

    if classifier_type == "lightgbm":
        if LGBMClassifier is None:
            raise ImportError("lightgbm is required for classifier_type='lightgbm'.")
        return LGBMClassifier(random_state=seed)

    return LogisticRegression(max_iter=1000, random_state=seed)


def get_profit_model(model_type, seed):
    if model_type == "xgboost":
        if XGBRegressor is None:
            raise ImportError("xgboost is required for profit_model_type='xgboost'.")
        return XGBRegressor(
            n_estimators=500,
            learning_rate=0.01,
            objective="reg:squarederror",
            random_state=seed,
        )

    return LinearRegression()


def prepare_screening_data(data, test_data, args):
    data_classification = data.copy()
    test_classification = test_data.copy()
    
    data_classification = data_classification.drop(columns=SCREENING_DROP_FEATURES, errors="ignore")
    test_classification = test_classification.drop(columns=SCREENING_DROP_FEATURES, errors="ignore")

    require_columns(data_classification, [args.target_col], "classification training data")
    require_columns(test_classification, [args.target_col], "classification test data")

    label_encoders, onehot_encoder = fit_encoders(
        data_classification,
        LABELENCODING_FEATURES,
        ONEHOT_FEATURES,
    )

    data_classification = encode_features(
        data_classification,
        label_encoders,
        onehot_encoder,
        LABELENCODING_FEATURES,
        ONEHOT_FEATURES,
    )
    test_classification = encode_features(
        test_classification,
        label_encoders,
        onehot_encoder,
        LABELENCODING_FEATURES,
        ONEHOT_FEATURES,
    )

    X_train = data_classification.drop(columns=args.target_col)
    y_train = data_classification[args.target_col]
    X_test = test_classification.drop(columns=args.target_col)
    y_test = test_classification[args.target_col]
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    return X_train, y_train, X_test, y_test, label_encoders, onehot_encoder


def load_synthetic_minority(args, label_encoders, onehot_encoder, train_columns):
    if args.synthetic_path is None or not os.path.exists(args.synthetic_path):
        print("Synthetic data not found. Screening classifier will use real training data only.")
        return None

    fake = pd.read_csv(args.synthetic_path, low_memory=True)
    require_columns(fake, [args.target_col], "synthetic data")

    if "term_months" in fake.columns:
        fake["term_months"] = fake["term_months"].apply(
            lambda x: 36 if abs(x - 36) < abs(x - 60) else 60
        )

    fake = fake[fake[args.target_col] == DEFAULT_LABEL].copy()
    fake_classification = fake.drop(columns=SCREENING_DROP_FEATURES, errors="ignore")
    fake_classification = encode_features(
        fake_classification,
        label_encoders,
        onehot_encoder,
        LABELENCODING_FEATURES,
        ONEHOT_FEATURES,
    )
    fake_classification = fake_classification.reindex(
        columns=list(train_columns) + [args.target_col],
        fill_value=0,
    )
    return fake_classification


def train_screening_gate(X_train, y_train, X_test, y_test, args, label_encoders, onehot_encoder):
    model = get_classifier(args.classifier_type, args.seed)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        random_state=args.seed,
        stratify=y_train,
    )

    train_dataset = pd.concat([X_tr, y_tr], axis=1)
    fake_classification = load_synthetic_minority(
        args,
        label_encoders,
        onehot_encoder,
        X_train.columns,
    )

    if fake_classification is not None and len(fake_classification) > 0:
        train_dataset = pd.concat([train_dataset, fake_classification], axis=0)
        train_dataset = shuffle(train_dataset, random_state=args.seed)

    X_fit = train_dataset.drop(columns=args.target_col)
    y_fit = train_dataset[args.target_col]

    if args.classifier_type == "xgboost":
        model.fit(X_fit, y_fit, eval_set=[(X_fit, y_fit), (X_val, y_val)], verbose=False)
    else:
        model.fit(X_fit, y_fit)

    default_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (default_prob >= args.tau).astype(int)
    screened_idx = default_prob < args.tau

    report = classification_report(
        y_test,
        y_pred,
        target_names=["Fully Paid", "Default"],
        output_dict=True,
        zero_division=0,
    )

    metrics = {
        "classifier_type": args.classifier_type,
        "tau": args.tau,
        "accuracy": accuracy_score(y_test, y_pred),
        "auroc": roc_auc_score(y_test, default_prob),
        "auprc": average_precision_score(y_test, default_prob),
        "brier_score": brier_score_loss(y_test, default_prob),
        "default_precision": report["Default"]["precision"],
        "default_recall": report["Default"]["recall"],
        "default_f1": report["Default"]["f1-score"],
        "fully_paid_precision": report["Fully Paid"]["precision"],
        "fully_paid_recall": report["Fully Paid"]["recall"],
        "fully_paid_f1": report["Fully Paid"]["f1-score"],
        "screened_candidates": int(screened_idx.sum()),
        "test_loans": int(len(X_test)),
    }

    screening_results = pd.DataFrame({
        "test_index": X_test.index,
        "default_probability": default_prob,
        "predicted_label": y_pred,
        "screened_candidate": screened_idx,
        "actual_label": y_test.values,
    })

    return model, default_prob, y_pred, screened_idx, metrics, screening_results


In [ ]:
def prepare_profit_data(data, test_data, args, label_encoders, onehot_encoder):
    train_regression = add_realized_outcomes(
        data,
        return_col=args.return_col,
        profit_col=args.profit_col,
    )
    test_regression = add_realized_outcomes(
        test_data,
        return_col=args.return_col,
        profit_col=args.profit_col,
    )

    require_columns(train_regression, [args.return_col, args.profit_col], "profit training data")
    require_columns(test_regression, [args.return_col, args.profit_col], "profit test data")
    
    train_regression = train_regression.drop(columns=SCREENING_DROP_FEATURES, errors="ignore")
    test_regression = test_regression.drop(columns=SCREENING_DROP_FEATURES, errors="ignore")

    train_regression_model = train_regression.drop(columns=PROFIT_DROP_FEATURES, errors="ignore")
    test_regression_model = test_regression.drop(columns=PROFIT_DROP_FEATURES, errors="ignore")

    train_regression_model = encode_features(
        train_regression_model,
        label_encoders,
        onehot_encoder,
        LABELENCODING_FEATURES,
        ONEHOT_FEATURES,
    )
    test_regression_model = encode_features(
        test_regression_model,
        label_encoders,
        onehot_encoder,
        LABELENCODING_FEATURES,
        ONEHOT_FEATURES,
    )

    y_train_return = train_regression[args.return_col]
    X_train = train_regression_model.reindex(index=y_train_return.index)
    valid_train = y_train_return.notna()

    X_train = X_train.loc[valid_train]
    y_train_return = y_train_return.loc[valid_train]

    X_test = test_regression_model.reindex(columns=X_train.columns, fill_value=0)
    return X_train, y_train_return, X_test, test_regression


def train_profit_model(data, test_data, screened_idx, args, label_encoders, onehot_encoder):
    X_train, y_train, X_test, test_regression = prepare_profit_data(
        data,
        test_data,
        args,
        label_encoders,
        onehot_encoder,
    )

    model = get_profit_model(args.profit_model_type, args.seed)

    if args.profit_model_type == "mlr":
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_screened_scaled = scaler.transform(X_test.loc[screened_idx])
        model.fit(X_train_scaled, y_train)
        predicted_return = model.predict(X_screened_scaled)
    else: # xgboost regressor.
        model.fit(X_train, y_train)
        predicted_return = model.predict(X_test.loc[screened_idx])

    screened_loans = test_regression.loc[screened_idx].copy()
    screened_loans["predicted_return"] = predicted_return
    screened_loans["predicted_profit"] = predicted_return * screened_loans["funded_amnt_inv"]
    return model, screened_loans


def get_top_percent_portfolio(candidate_df, top_percent):
    candidate_df = candidate_df.copy()
    top_n = int(len(candidate_df) * top_percent)
    top_n = max(top_n, 1)
    top_indices = (
        candidate_df
        .sort_values(by="predicted_return", ascending=False)
        .head(top_n)
    )
    return top_indices


def evaluate_portfolio(portfolio_df, ratio, args):
    total_count = len(portfolio_df)
    success_ratio = np.nan
    default_ratio = np.nan

    if args.target_col in portfolio_df.columns and total_count > 0:
        success_ratio = (portfolio_df[args.target_col] == 0).mean()
        default_ratio = (portfolio_df[args.target_col] == DEFAULT_LABEL).mean()

    return {
        "top_k_ratio": ratio,
        "number_selected": total_count,
        "mean_realized_return": portfolio_df[args.return_col].mean(),
        "mean_realized_profit": portfolio_df[args.profit_col].mean(),
        "total_realized_profit": portfolio_df[args.profit_col].sum(),
        "success_ratio": success_ratio,
        "default_ratio": default_ratio,
    }


def construct_top_k_portfolios(screened_loans, args):
    if screened_loans.empty:
        raise ValueError("No loans passed the screening gate. Increase tau or check classifier output.")

    portfolio_summary = []
    portfolios = {}

    for ratio in args.top_k_ratios:
        top_portfolio = get_top_percent_portfolio(screened_loans, ratio)
        portfolios[ratio] = top_portfolio
        portfolio_summary.append(evaluate_portfolio(top_portfolio, ratio, args))

    return portfolios, pd.DataFrame(portfolio_summary)


def save_results(screening_results, screening_metrics, portfolios, summary_df, args):
    os.makedirs(args.results_dir, exist_ok=True)

    screening_results.to_csv(
        os.path.join(args.results_dir, "screening_results.csv"),
        index=False,
    )

    screening_metrics_df = pd.DataFrame([screening_metrics])
    summary_metrics = pd.concat(
        [
            screening_metrics_df.assign(result_type="screening"),
            summary_df.assign(result_type="portfolio"),
        ],
        axis=0,
        ignore_index=True,
        sort=False,
    )
    summary_metrics.to_csv(
        os.path.join(args.results_dir, "summary_metrics.csv"),
        index=False,
    )

    for ratio, portfolio_df in portfolios.items():
        ratio_label = int(round(ratio * 100))
        portfolio_df.to_csv(
            os.path.join(args.results_dir, f"portfolio_top{ratio_label}.csv"),
            index=True,
        )

## 3. Run the Two-stage Pipeline


In [ ]:
np.random.seed(args.seed)

data = pd.read_csv(args.train_path, low_memory=True)
test_data = pd.read_csv(args.test_path, low_memory=True)

require_columns(data, [args.target_col], "training data")
require_columns(test_data, [args.target_col], "test data")

X_train, y_train, X_test, y_test, label_encoders, onehot_encoder = prepare_screening_data(
    data,
    test_data,
    args,
)

_, default_prob, y_pred, screened_idx, screening_metrics, screening_results = train_screening_gate(
    X_train,
    y_train,
    X_test,
    y_test,
    args,
    label_encoders,
    onehot_encoder,
)

_, screened_loans = train_profit_model(
    data,
    test_data,
    screened_idx,
    args,
    label_encoders,
    onehot_encoder,
)

portfolios, portfolio_summary = construct_top_k_portfolios(screened_loans, args)
save_results(screening_results, screening_metrics, portfolios, portfolio_summary, args)

print("Two-stage decision-support pipeline completed.")
print(f"Classifier: {args.classifier_type} | Profit model: {args.profit_model_type}")
print(f"Screening threshold tau: {args.tau}")
print(f"Screened candidates: {screening_metrics['screened_candidates']} / {screening_metrics['test_loans']}")
print("Classification metrics:")
print(pd.DataFrame([screening_metrics]).round(4).to_string(index=False))
print("Portfolio summary:")
print(portfolio_summary.round(4).to_string(index=False))
print(f"Saved CSV files under: {args.results_dir}")

## 4. Inspect Outputs


In [ ]:
pd.DataFrame([screening_metrics]).round(4)

In [ ]:
portfolio_summary.round(4)

In [ ]:
screening_results.head()

In [ ]:
{f"Top {int(ratio * 100)}%": df.head() for ratio, df in portfolios.items()}